# module-base-class-custom — worked example 2: Recursive train/eval over submodules

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-base-class-custom`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Adding a `.training` flag and `.train(mode)`/`.eval()` methods that recurse over `_modules` makes a mode toggle propagate through the whole module tree. Both methods return `self` so chaining like `model.eval()` works in expressions.

## Worked solution

We extend the base Module with a recursive training toggle.

1. **Bootstrap training.** `__init__` installs `training = True` via `object.__setattr__`, alongside the registries.
2. **train(mode).** Sets `self.training = mode`, then calls `m.train(mode)` for each submodule, so the flag flips everywhere. Returns `self`.
3. **eval().** Just `self.train(False)`, returning self.
4. **Verify propagation.** After `net.eval()`, the root and every nested submodule report `training == False`; after `net.train()`, all are True again.

The demo toggles a nested model and prints the training flags of the root and a deep child before and after eval/train.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
        object.__setattr__(self, 'training', True)
    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
        elif isinstance(value, Module):
            self._modules[name] = value
        object.__setattr__(self, name, value)
    def train(self, mode=True):
        self.training = mode
        for m in self._modules.values():
            m.train(mode)
        return self
    def eval(self):
        return self.train(False)

class Inner(Module):
    def __init__(self):
        super().__init__()
        self.w = Parameter(np.zeros(2))

class Outer(Module):
    def __init__(self):
        super().__init__()
        self.inner = Inner()

net = Outer()
print('before:', net.training, net.inner.training)
net.eval()
print('after eval:', net.training, net.inner.training)
ret = net.train()
print('after train:', net.training, net.inner.training, '| chains:', ret is net)